# TFG V7 — Quantum walk with local defects

This version reformulates the search as a continuous-time quantum walk on the candidate window-start graph. Valid windows are marked as local energy defects in the Hamiltonian.

## Problem model

Each candidate start is a graph node. Two starts are adjacent when their ND coordinates differ by one step in exactly one dimension. Valid windows are the nodes whose associated window is entirely free:

```text
P_V = sum_{i valid} |i><i|
```

The notebook compares adjacency and Laplacian Hamiltonian conventions for the same graph.

## Circuit structure

1. Build the candidate-start graph and its Hamiltonian.
2. Add a defect term on the valid-window subspace.
3. Start from the uniform superposition over candidate starts.
4. Search over evolution time and defect strength.
5. Compile the selected exact evolution as a dense unitary for small cases.
6. Compare success probability, spectra, time curves, and per-case summaries.

## Registers and data

- `i`: candidate window-start index register.
- `A`, `L`: adjacency and Laplacian graph matrices.
- `P_V`: diagonal projector onto valid windows.
- `gamma`, `lambda`, `t`: quantum-walk parameters.

## Purpose of this version

V7 tests the defect quantum-walk model as an exact spectral prototype. It uses classically known valid indices to build the Hamiltonian, so it is mainly a diagnostic and benchmarking step before the hardware-oriented V7.1 circuit.

## Problem Definition

Given a $D$-dimensional binary grid of sizes

$$N=(N_1,\ldots,N_D),$$

where occupied cells are marked by 1 and free cells by 0, we search for start positions $i$ such that a contiguous window

$$M=(M_1,\ldots,M_D)$$

is entirely free. The set of candidate starts is

$$\prod_{d=1}^D \{0,\ldots,N_d-M_d\},$$

with total size

$$W=\prod_d (N_d-M_d+1).$$

These starts form a graph: two starts are adjacent if their coordinates differ by exactly 1 in exactly one dimension. The graph is a line in 1D, a rectangular lattice in 2D, and a Cartesian grid in higher dimensions.


## Theoretical Background

### Discrete-time vs continuous-time quantum walks

A **discrete-time quantum walk** usually alternates two operations: a coin operation and a shift operation. The coin decides which direction the walker should move, and the shift moves the walker through the graph. This typically requires an explicit coin register.

A **continuous-time quantum walk** has no coin register. The walker evolves directly under a Hamiltonian,

$$|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle.$$

In this notebook the Hilbert space is the candidate window-start graph. We compare two standard Hamiltonian conventions for the same graph:

$$H_{\mathrm{adj}} = -\gamma A - \lambda P_V,$$

and

$$H_{\mathrm{lap}} = \gamma L - \lambda P_V, \quad L=D-A.$$

The adjacency convention is common in spatial-search CTQW algorithms because the uniform-like mode is a low-energy reference on regular graphs. The Laplacian convention follows the transition-rate CTQW definition used in many transport papers. On regular graphs they differ only by a global phase; on finite window-start grids, boundary nodes have smaller degree, so the two conventions can produce different dynamics.

### Defect mechanism

The valid all-zero windows are marked by adding a local energy perturbation:

$$P_V = \sum_{k\in V}|k\rangle\langle k|,$$

where $V$ is the set of valid window indices. The negative diagonal term $-\lambda P_V$ lowers the energy of valid nodes. Spectrally, this can pull bound states out of the bulk band and localise amplitude near the marked nodes. Starting from a uniform superposition, the walk can transfer probability into the valid subspace at special times.


## Requirements and Imports

This cell imports the numerical, plotting, and Qiskit objects used throughout the notebook. The plotting backend is configured for file generation because all figures are saved to disk.


In [1]:

# pip install qiskit qiskit-aer qiskit-ibm-runtime numpy matplotlib

import os
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(os.getcwd(), ".cache"))
os.environ.setdefault("MPLBACKEND", "Agg")

from math import prod
from pathlib import Path
import csv
import re
import time

import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import MCXGate, PhaseGate, UnitaryGate

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

VALID_COLOR = "#2ecc71"
INVALID_COLOR = "#e74c3c"
BASELINE_COLOR = "0.45"

V7_OUTPUT_DIR = Path("TFG_V7_Analysis")
V7_OUTPUT_DIR.mkdir(exist_ok=True)

# V7 uses the earliest time whose success probability is within this
# absolute tolerance of the best scanned peak. This keeps the walk fast
# while preserving a strong success-probability advantage.
V7_PEAK_TOLERANCE = 0.05

# If True, each case scans only times t <= r_Grover = (pi/4)*sqrt(W/K).
# This forces the quantum-walk search to use no more time/iteration scale
# than the standard Grover estimate for the same valid fraction.
V7_ENFORCE_GROVER_TIME_CAP = True

V7_DEFAULT_T_MAX = 200.0
V7_T_STEPS = 4000

print("qiskit version:", qiskit.__version__)


qiskit version: 2.3.1


## Utility Functions

The following utility functions are copied directly from `TFG_V4.ipynb`. They define the grid geometry, the classical cost function, the valid-window set, Gray ordering, and the initial uniform superposition over valid index states.


In [2]:

# =========================================================
# ND geometry and classical utilities
# =========================================================

def validate_problem(N, M):
    if len(N) != len(M):
        raise ValueError("N and M must have the same dimension.")
    for d, (n_d, m_d) in enumerate(zip(N, M)):
        if n_d <= 0 or m_d <= 0:
            raise ValueError(f"N[{d}] and M[{d}] must be positive.")
        if m_d > n_d:
            raise ValueError(f"M[{d}] cannot be greater than N[{d}].")


def coord_to_index(coord, dims):
    """Converts ND coordinates to a row-major linear index.
    Example with dims = [4,4]:
    (0,0) -> 0    (0,1) -> 1    (0,2) -> 2    (0,3) -> 3
    (1,0) -> 4    (1,1) -> 5    (1,2) -> 6    (1,3) -> 7
    (2,0) -> 8    (2,1) -> 9    (2,2) -> 10   (2,3) -> 11
    (3,0) -> 12   (3,1) -> 13   (3,2) -> 14   (3,3) -> 15
    """
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin


def index_to_coord(index, dims):
    """Converts a row-major linear index to ND coordinates.
    Example with dims = [4,4]:
    0 -> (0,0)    1 -> (0,1)    2 -> (0,2)    3 -> (0,3)
    4 -> (1,0)    5 -> (1,1)    6 -> (1,2)    7 -> (1,3)
    8 -> (2,0)    9 -> (2,1)    10 -> (2,2)   11 -> (2,3)
    12 -> (3,0)   13 -> (3,1)    14 -> (3,2)   15 -> (3,3)
    """
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)


def valid_starts_nd(N, M):
    """Valid starting coordinates for a window M inside N."""
    return list(np.ndindex(tuple(N[d] - M[d] + 1 for d in range(len(N)))))


def window_qubits_nd(start, N, M):
    """Linear indices of the cells covered by the window starting at start."""
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits


def normalize_coord(coord, D):
    if D == 1 and isinstance(coord, int):
        return (coord,)
    return tuple(coord)


def build_grid_bits(N, occupied_coords):
    """Returns a classical vector with 1 on occupied cells and 0 on free cells."""
    D = len(N)
    grid = [0] * prod(N)
    for raw_coord in occupied_coords:
        coord = normalize_coord(raw_coord, D)
        if len(coord) != D:
            raise ValueError(f"Coordinate {coord} does not have dimension {D}.")
        for d, x in enumerate(coord):
            if x < 0 or x >= N[d]:
                raise ValueError(f"Coordinate {coord} is outside the grid N={N}.")
        grid[coord_to_index(coord, N)] = 1
    return grid


def compute_window_cost_classical(grid_bits, start, N, M):
    """C(i): number of ones in the window associated with start."""
    return sum(grid_bits[q] for q in window_qubits_nd(start, N, M))


def get_valid_indices(grid_bits, starts, N, M):
    return [i for i, start in enumerate(starts) if compute_window_cost_classical(grid_bits, start, N, M) == 0]


def gray_order_valid(W, IDX):
    """
    000 -> 0
    001 -> 1
    011 -> 3
    010 -> 2
    110 -> 6
    111 -> 7
    101 -> 5
    100 -> 4
    """
    gray_full = [t ^ (t >> 1) for t in range(2**IDX)]
    return [g for g in gray_full if g < W]


def format_nd_array_from_bits(bitstring, dims):
    arr = np.array(list(bitstring), dtype=str).reshape(tuple(dims))
    return np.array2string(arr, separator=' ').replace("'", "")


In [3]:

# =========================================================
# Circuit index construction utility
# =========================================================

def prepare_valid_index_superposition(qc, idx, W):
    IDX = len(idx)
    amps = np.zeros(2**IDX, dtype=complex)
    amps[:W] = 1 / np.sqrt(W)
    qc.initialize(amps, idx)


## Case Studies

The first ten case studies reproduce the original benchmark set: five 1D cases, four 2D cases, and one 3D case. Cases 11--20 add larger, more structured clustered layouts that mimic grouped obstacles, corridor-like free regions, rectangular windows, and small 3D volumes. This keeps the original comparisons intact while testing whether the quantum-walk defect model benefits from richer geometry.


In [4]:

CASE_STUDIES = [
    {
        "name": "01_1d_tiny_single_gap",
        "description": "Minimal 1D case with one valid window.",
        "N": [6], "M": [2],
        "occupied_coords": [(0,), (3,), (4,)],
        "theta": np.pi / 2, "beta": 0.30, "mixer_method": "local_geometric",
    },
    {
        "name": "02_1d_main_reference",
        "description": "Reference 1D instance used in the manual main experiment.",
        "N": [8], "M": [2],
        "occupied_coords": [(0,), (1,), (2,), (6,), (7,)],
        "theta": np.pi / 3, "beta": 0.60, "mixer_method": "local_geometric",
    },
    {
        "name": "03_1d_two_free_regions",
        "description": "1D grid with two occupied blocks and a central valid plateau.",
        "N": [10], "M": [3],
        "occupied_coords": [(0,), (1,), (7,), (8,), (9,)],
        "theta": np.pi / 3, "beta": 0.30, "mixer_method": "local_geometric",
    },
    {
        "name": "04_1d_clustered_medium",
        "description": "Medium 1D case with several clustered obstacles.",
        "N": [16], "M": [3],
        "occupied_coords": [(0,), (1,), (5,), (6,), (7,), (13,), (14,)],
        "theta": np.pi / 3, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "05_1d_long_clustered_blocks",
        "description": "Longer 1D case with separated occupied blocks.",
        "N": [32], "M": [4],
        "occupied_coords": [(0,), (1,), (2,), (9,), (10,), (11,), (18,), (19,), (28,), (29,), (30,), (31,)],
        "theta": np.pi / 4, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "06_2d_tiny_corner_block",
        "description": "Small 2D case with a single valid 2x2 region.",
        "N": [3, 3], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 2), (2, 2)],
        "theta": np.pi / 2, "beta": 0.28, "mixer_method": "local_geometric",
    },
    {
        "name": "07_2d_small_diagonal_block",
        "description": "4x4 grid with diagonal obstacles and one clear lower-left solution.",
        "N": [4, 4], "M": [2, 2],
        "occupied_coords": [(1, 1), (2, 2), (0, 3)],
        "theta": np.pi / 2, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "08_2d_medium_clustered_obstacles",
        "description": "5x5 grid with two compact occupied clusters.",
        "N": [5, 5], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (3, 3), (3, 4), (4, 3)],
        "theta": np.pi / 3, "beta": 0.22, "mixer_method": "local_geometric",
    },
    {
        "name": "09_2d_rectangular_window",
        "description": "6x6 grid with a non-square 3x2 window.",
        "N": [6, 6], "M": [3, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (4, 4), (4, 5), (5, 4), (2, 3)],
        "theta": np.pi / 3, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "10_3d_small_clustered_obstacles",
        "description": "Small 3D grid with two compact occupied clusters.",
        "N": [4, 4, 3], "M": [2, 2, 2],
        "occupied_coords": [(0, 0, 0), (0, 0, 1), (1, 0, 0), (3, 3, 2), (3, 2, 2), (2, 3, 2)],
        "theta": np.pi / 4, "beta": 0.16, "mixer_method": "local_geometric",
    },    {
        "name": "11_1d_dual_clustered_gaps",
        "description": "Long 1D layout with four compact obstacle clusters and two broad free corridors.",
        "N": [40], "M": [5],
        "occupied_coords": [(0,), (1,), (2,), (3,), (10,), (11,), (12,), (13,), (24,), (25,), (26,), (27,), (36,), (37,), (38,), (39,)],
        "theta": np.pi / 4, "beta": 0.16, "mixer_method": "local_geometric",
    },
    {
        "name": "12_1d_staggered_clustered_barriers",
        "description": "Long 1D case with staggered grouped barriers and several differently sized free regions.",
        "N": [48], "M": [6],
        "occupied_coords": [(0,), (1,), (2,), (8,), (9,), (10,), (11,), (20,), (21,), (22,), (23,), (34,), (35,), (36,), (37,), (45,), (46,), (47,)],
        "theta": np.pi / 4, "beta": 0.14, "mixer_method": "local_geometric",
    },
    {
        "name": "13_2d_corridor_clusters",
        "description": "2D map with clustered blocks that leave corridor-like free regions.",
        "N": [6, 6], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (2, 3), (2, 4), (3, 3), (4, 1), (5, 4), (5, 5), (4, 5)],
        "theta": np.pi / 3, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "14_2d_rooms_and_walls",
        "description": "2D room-like layout with short wall segments and a rectangular window.",
        "N": [7, 7], "M": [2, 3],
        "occupied_coords": [(1, 1), (1, 2), (1, 3), (3, 0), (3, 1), (3, 2), (4, 5), (5, 5), (5, 6), (6, 0), (6, 1)],
        "theta": np.pi / 3, "beta": 0.16, "mixer_method": "local_geometric",
    },
    {
        "name": "15_2d_grouped_islands",
        "description": "Larger 2D grid with several compact obstacle islands and many correlated valid windows.",
        "N": [8, 8], "M": [3, 3],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (2, 5), (2, 6), (3, 5), (5, 2), (6, 2), (6, 3), (7, 6), (7, 7)],
        "theta": np.pi / 4, "beta": 0.14, "mixer_method": "local_geometric",
    },
    {
        "name": "16_2d_narrow_passages",
        "description": "Rectangular 2D grid with grouped barriers that create narrow passage structure.",
        "N": [9, 7], "M": [3, 2],
        "occupied_coords": [(1, 1), (2, 1), (3, 1), (4, 1), (1, 5), (2, 5), (6, 4), (7, 4), (8, 4), (5, 2), (5, 3)],
        "theta": np.pi / 4, "beta": 0.14, "mixer_method": "local_geometric",
    },
    {
        "name": "17_2d_large_clustered_office",
        "description": "Office-like 2D floor plan with separated block clusters and repeated free pockets.",
        "N": [9, 9], "M": [3, 3],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (1, 1), (3, 3), (3, 4), (4, 3), (6, 1), (7, 1), (7, 2), (5, 6), (6, 6), (6, 7), (8, 8)],
        "theta": np.pi / 4, "beta": 0.12, "mixer_method": "local_geometric",
    },
    {
        "name": "18_2d_rectangular_window_clusters",
        "description": "Larger 2D case with a 2x4 window and shelf-like clustered obstacle bands.",
        "N": [10, 8], "M": [2, 4],
        "occupied_coords": [(0, 0), (0, 1), (0, 2), (2, 5), (2, 6), (3, 5), (5, 1), (5, 2), (6, 1), (8, 4), (8, 5), (9, 4), (9, 5)],
        "theta": np.pi / 4, "beta": 0.12, "mixer_method": "local_geometric",
    },
    {
        "name": "19_3d_dual_room_clusters",
        "description": "3D volume with two compact obstacle rooms and one smaller internal cluster.",
        "N": [5, 5, 4], "M": [2, 2, 2],
        "occupied_coords": [(0, 0, 0), (0, 0, 1), (1, 0, 0), (1, 1, 0), (3, 3, 2), (3, 3, 3), (4, 3, 2), (4, 4, 3), (2, 1, 1), (2, 2, 1)],
        "theta": np.pi / 5, "beta": 0.10, "mixer_method": "local_geometric",
    },
    {
        "name": "20_3d_layered_obstacle_bands",
        "description": "3D layered layout with band-like clustered obstacles across different depths.",
        "N": [6, 5, 4], "M": [2, 2, 2],
        "occupied_coords": [(0, 0, 0), (0, 1, 0), (1, 0, 0), (2, 3, 1), (2, 4, 1), (3, 3, 1), (4, 1, 2), (5, 1, 2), (5, 2, 2), (1, 4, 3), (2, 4, 3)],
        "theta": np.pi / 5, "beta": 0.10, "mixer_method": "local_geometric",
    },
]


## Context Builders and Plot Helpers

The context builder converts each case into the classical data needed by the quantum walk: starts, costs, valid indices, $W$, and the uniform success baseline $K/W$. The plotting helper saves each figure as both PDF and PNG.


In [5]:

def qw_slug(text):
    """Return a filesystem-safe lowercase slug for figure filenames."""
    return re.sub(r"[^a-zA-Z0-9_]+", "_", str(text)).strip("_").lower()


def save_qw_figure(fig, stem):
    """Save a Matplotlib figure as PDF and PNG in the V7 analysis folder."""
    # Save every figure in the V7 experiment folder.
    V7_OUTPUT_DIR.mkdir(exist_ok=True)
    fig.savefig(V7_OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(V7_OUTPUT_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def qw_case_context(case):
    """Build the classical search-space context for one case study."""
    # Validate dimensions and precompute classical window costs.
    N_case = list(case["N"])
    M_case = list(case["M"])
    occupied_case = list(case["occupied_coords"])
    validate_problem(N_case, M_case)
    starts_case = valid_starts_nd(N_case, M_case)
    grid_bits_case = build_grid_bits(N_case, occupied_case)
    costs_case = [
        compute_window_cost_classical(grid_bits_case, s, N_case, M_case)
        for s in starts_case
    ]
    valid_indices_case = [i for i, c in enumerate(costs_case) if c == 0]
    if not valid_indices_case:
        raise ValueError(f"Case {case['name']} has no valid windows; change occupied_coords.")
    W_case = len(starts_case)
    IDX_case = int(np.ceil(np.log2(W_case))) if W_case > 1 else 1
    if W_case > 2**IDX_case:
        raise ValueError(f"W={W_case} does not fit in IDX={IDX_case} qubits.")
    return {
        "name": case["name"],
        "description": case.get("description", ""),
        "N": N_case,
        "M": M_case,
        "occupied_coords": occupied_case,
        "starts": starts_case,
        "grid_bits": grid_bits_case,
        "costs": costs_case,
        "valid_indices": valid_indices_case,
        "W": W_case,
        "K": len(valid_indices_case),
        "IDX": IDX_case,
        "P_uniform": len(valid_indices_case) / W_case,
    }


CASE_CONTEXTS = [qw_case_context(case) for case in CASE_STUDIES]
print(f"Loaded {len(CASE_CONTEXTS)} case contexts.")
for ctx in CASE_CONTEXTS:
    print(f"{ctx['name']} | W(Possible starts)={ctx['W']} | K(Valid starts)={ctx['K']} | P_uniform={ctx['P_uniform']:.4f}")


Loaded 20 case contexts.
01_1d_tiny_single_gap | W(Possible starts)=5 | K(Valid starts)=1 | P_uniform=0.2000
02_1d_main_reference | W(Possible starts)=7 | K(Valid starts)=2 | P_uniform=0.2857
03_1d_two_free_regions | W(Possible starts)=8 | K(Valid starts)=3 | P_uniform=0.3750
04_1d_clustered_medium | W(Possible starts)=14 | K(Valid starts)=4 | P_uniform=0.2857
05_1d_long_clustered_blocks | W(Possible starts)=29 | K(Valid starts)=11 | P_uniform=0.3793
06_2d_tiny_corner_block | W(Possible starts)=4 | K(Valid starts)=1 | P_uniform=0.2500
07_2d_small_diagonal_block | W(Possible starts)=9 | K(Valid starts)=1 | P_uniform=0.1111
08_2d_medium_clustered_obstacles | W(Possible starts)=16 | K(Valid starts)=9 | P_uniform=0.5625
09_2d_rectangular_window | W(Possible starts)=20 | K(Valid starts)=8 | P_uniform=0.4000
10_3d_small_clustered_obstacles | W(Possible starts)=18 | K(Valid starts)=12 | P_uniform=0.6667
11_1d_dual_clustered_gaps | W(Possible starts)=36 | K(Valid starts)=12 | P_uniform=0.3333


## Quantum-Walk Core Functions

These functions build the window-start graph, its Laplacian, the defect Hamiltonian, and the exact continuous-time evolution. The time search is classical and uses one Hermitian diagonalisation followed by a fully vectorised evaluation of $P_{valid}(t)$.


In [6]:
def build_adjacency_matrix(starts, N):
    """Build the symmetric adjacency matrix of the window-start graph.

    Two window starts are adjacent when they differ by exactly one lattice
    step in one coordinate and are equal in all other coordinates. The
    dictionary lookup makes neighbour discovery O(1) per candidate neighbour.
    """
    # Map coordinates to indices for constant-time neighbor lookup.
    starts = [tuple(start) for start in starts]
    W = len(starts)
    D = len(N)
    start_to_index = {start: i for i, start in enumerate(starts)}
    A = np.zeros((W, W), dtype=np.float64)

    for i, start in enumerate(starts):
        for d in range(D):
            for step in (-1, 1):
                neighbour = list(start)
                neighbour[d] += step
                neighbour = tuple(neighbour)
                j = start_to_index.get(neighbour)
                if j is not None:
                    A[i, j] = 1.0
                    A[j, i] = 1.0

    return A


def build_graph_laplacian(A):
    """Return the unnormalised and normalised graph Laplacians.

    The unnormalised Laplacian is L = D - A, where D is the diagonal degree
    matrix. The normalised Laplacian is D^{-1/2} L D^{-1/2}, with zero inverse
    degree assigned to isolated nodes.
    """
    # Compute degrees and avoid division by zero on isolated nodes.
    A = np.asarray(A, dtype=np.float64)
    degrees = A @ np.ones(A.shape[0], dtype=np.float64)
    L = np.diag(degrees) - A
    inv_sqrt_degrees = np.zeros_like(degrees)
    nonzero = degrees > 0
    inv_sqrt_degrees[nonzero] = 1.0 / np.sqrt(degrees[nonzero])
    D_inv_sqrt = np.diag(inv_sqrt_degrees)
    L_norm = D_inv_sqrt @ L @ D_inv_sqrt
    return L, L_norm


def hamiltonian_mode_label(mode):
    """Return a readable label for the selected CTQW Hamiltonian convention."""
    # Centralize labels so plots and tables use the same text.
    labels = {
        "adjacency": "Adjacency convention: H = -gamma * A - lambda * P_V",
        "laplacian": "Laplacian convention: H = gamma * L - lambda * P_V",
    }
    try:
        return labels[str(mode).lower()]
    except KeyError as exc:
        raise ValueError(f"Unknown Hamiltonian mode: {mode!r}") from exc


def build_defect_hamiltonian(A, valid_indices, lam, gamma=1.0, mode="adjacency"):
    """Build the CTQW defect Hamiltonian for one convention.

    ``mode='adjacency'`` uses the spatial-search convention
    H = -gamma*A - lambda*P_V. ``mode='laplacian'`` uses the transition-rate
    convention H = gamma*L - lambda*P_V, where L=D-A.
    """
    # Copy the base matrix so A is not modified while marking defects.
    mode = str(mode).lower()
    A = np.asarray(A, dtype=np.float64)
    if mode == "adjacency":
        H = -float(gamma) * A.copy()
    elif mode == "laplacian":
        L, _ = build_graph_laplacian(A)
        H = float(gamma) * L.copy()
    else:
        raise ValueError(f"Unknown Hamiltonian mode: {mode!r}")

    for k in valid_indices:
        H[int(k), int(k)] -= float(lam)

    if not np.allclose(H, H.conj().T, atol=1e-12):
        raise ValueError("H_defect is not Hermitian.")
    return H


def find_optimal_time(H, valid_indices, W, t_max=200, t_steps=4000, peak_tolerance=0.0):
    """Find a high-quality time for the valid-subspace probability.

    The Hamiltonian is diagonalised once with np.linalg.eigh. The full time
    curve is evaluated by broadcasting exp(-i E_k t). If peak_tolerance > 0,
    the returned t_star is the earliest sampled time whose probability is
    within peak_tolerance of the maximum. This avoids selecting very late
    recurrences when an almost equally good first peak exists.
    """
    # Trivial case: one node. If it is valid, the probability is constant.
    if W == 1:
        t_values = np.linspace(0.0, float(t_max), int(t_steps))
        p_value = 1.0 if 0 in set(valid_indices) else 0.0
        return 0.0, p_value, t_values, np.full_like(t_values, p_value, dtype=float)

    valid_indices = list(map(int, valid_indices))
    if not valid_indices:
        raise ValueError("valid_indices must not be empty.")

    eigenvalues, eigenvectors = np.linalg.eigh(np.asarray(H, dtype=np.complex128))
    psi0 = np.ones(int(W), dtype=complex) / np.sqrt(W)
    coeffs = eigenvectors.conj().T @ psi0
    t_values = np.linspace(0.0, float(t_max), int(t_steps))

    phases = np.exp(-1j * eigenvalues[:, None] * t_values[None, :])
    psi_t = eigenvectors @ (phases * coeffs[:, None])
    p_valid_curve = np.sum(np.abs(psi_t[valid_indices, :])**2, axis=0)

    max_idx = int(np.argmax(p_valid_curve))
    max_p = float(p_valid_curve[max_idx])
    if peak_tolerance and peak_tolerance > 0:
        threshold = max(0.0, max_p - float(peak_tolerance))
        candidate_indices = np.flatnonzero(p_valid_curve >= threshold)
        best_idx = int(candidate_indices[0]) if len(candidate_indices) else max_idx
    else:
        best_idx = max_idx

    t_star = float(t_values[best_idx])
    p_star = float(p_valid_curve[best_idx])
    return t_star, p_star, t_values, p_valid_curve


def scan_lambda(H_adj, valid_indices, W, lambda_values=None, t_max=200, t_steps=4000,
                gamma=1.0, mode="adjacency", peak_tolerance=V7_PEAK_TOLERANCE):
    """Scan defect strengths and return (lambda, t_star, p_star) rows."""
    # The V7 grid mixes linear and geometric values so narrow resonances are not missed.
    if lambda_values is None:
        lambda_values = v7_lambda_grid(W) if "v7_lambda_grid" in globals() else np.linspace(0.1, 5.0 * np.sqrt(W), 30)
    rows = []
    for lam in lambda_values:
        H = build_defect_hamiltonian(
            H_adj, valid_indices, lam=float(lam), gamma=gamma, mode=mode
        )
        t_star, p_star, _t_values, _p_curve = find_optimal_time(
            H, valid_indices, W, t_max=t_max, t_steps=t_steps, peak_tolerance=peak_tolerance
        )
        rows.append((float(lam), float(t_star), float(p_star)))
    return rows


## Unit Tests for the Graph Construction

Before using the graph as a Hamiltonian, we check the two basic structural properties required of an undirected simple graph: symmetry and zero diagonal.


In [7]:

for ctx in CASE_CONTEXTS:
    A_test = build_adjacency_matrix(ctx["starts"], ctx["N"])
    assert A_test.shape == (ctx["W"], ctx["W"])
    assert np.allclose(A_test, A_test.T), f"Adjacency is not symmetric for {ctx['name']}"
    assert np.allclose(np.diag(A_test), 0.0), f"Adjacency diagonal is not zero for {ctx['name']}"
print("Adjacency unit tests passed for all case studies.")


Adjacency unit tests passed for all case studies.


## Exact Quantum Circuit Construction

The circuit uses only one quantum register, `i`, holding the window-start index. The continuous-time evolution is compiled exactly as a dense `UnitaryGate` after diagonalising the defect Hamiltonian. States outside the valid index range $0,\ldots,W-1$ remain untouched by an identity block.


In [8]:
def build_quantum_walk_circuit(N, M, occupied_coords, lam, t_star, gamma=1.0, mode="adjacency"):
    """Build the exact continuous-time defect quantum-walk circuit.

    The circuit prepares the uniform superposition over W candidate starts,
    embeds exp(-i H_defect t_star) into the 2**IDX-dimensional index register,
    and returns both the circuit and metadata needed for analysis.
    """
    # Build the full classical problem data needed to compile the exact Hamiltonian.
    validate_problem(N, M)
    starts = valid_starts_nd(N, M)
    W = len(starts)
    IDX = int(np.ceil(np.log2(W))) if W > 1 else 1
    if W > 2**IDX:
        raise ValueError(f"W={W} does not fit in IDX={IDX} qubits.")

    grid_bits = build_grid_bits(N, occupied_coords)
    costs = [compute_window_cost_classical(grid_bits, start, N, M) for start in starts]
    valid_indices = [i for i, cost in enumerate(costs) if cost == 0]
    if not valid_indices:
        raise ValueError("The instance has no valid all-zero windows.")

    A = build_adjacency_matrix(starts, N)
    H_defect = build_defect_hamiltonian(A, valid_indices, lam=lam, gamma=gamma, mode=mode)

    eigenvalues, eigenvectors = np.linalg.eigh(H_defect)
    phases = np.exp(-1j * eigenvalues * float(t_star))
    U_walk_W = eigenvectors @ np.diag(phases) @ eigenvectors.conj().T

    dim_full = 2**IDX
    U_full = np.eye(dim_full, dtype=complex)
    U_full[:W, :W] = U_walk_W
    assert np.allclose(U_full @ U_full.conj().T, np.eye(dim_full), atol=1e-8)

    idx = QuantumRegister(IDX, "i")
    qc = QuantumCircuit(idx)
    prepare_valid_index_superposition(qc, idx, W)
    if W > 1:
        short_label = "Adj" if str(mode).lower() == "adjacency" else "Lap"
        label = f"QW({short_label}, t={float(t_star):.2f}, lam={float(lam):.3f})"
        qc.append(UnitaryGate(U_full, label=label), list(idx))

    metadata = {
        "N": list(N),
        "M": list(M),
        "occupied_coords": [normalize_coord(c, len(N)) for c in occupied_coords],
        "W": W,
        "K": len(valid_indices),
        "IDX": IDX,
        "valid_indices": valid_indices,
        "starts": starts,
        "grid_bits": grid_bits,
        "costs": costs,
        "lam": float(lam),
        "gamma": float(gamma),
        "mode": str(mode).lower(),
        "t_star": float(t_star),
        "P_uniform": len(valid_indices) / W,
        "H_defect": H_defect,
        "A": A,
    }
    return qc, metadata


def index_probs_from_statevector(sv, W, IDX):
    """Extract probabilities over the first W index states from a statevector."""
    # The quantum register only contains idx; the first W states are the physical candidates.
    data = np.asarray(sv.data, dtype=complex)
    outside_weight = float(np.sum(np.abs(data[W:])**2)) if W < len(data) else 0.0
    if outside_weight >= 1e-10:
        raise ValueError(f"Unexpected probability {outside_weight:.3e} in padded index states.")
    return np.abs(data[:W])**2


def run_qw_case(ctx, lam=None, gamma=1.0, mode="adjacency", t_max=200, t_steps=4000,
                peak_tolerance=V7_PEAK_TOLERANCE):
    """Run the full exact-defect quantum-walk pipeline for one case context."""
    # If lambda is not provided, use the initial sqrt(W)*gamma heuristic.
    W = int(ctx["W"])
    valid_indices = list(ctx["valid_indices"])
    A = build_adjacency_matrix(ctx["starts"], ctx["N"])
    if lam is None:
        lam = np.sqrt(W) * float(gamma)
    H_defect = build_defect_hamiltonian(
        A, valid_indices, lam=lam, gamma=gamma, mode=mode
    )
    t_star, p_star, t_values, p_valid_curve = find_optimal_time(
        H_defect, valid_indices, W, t_max=t_max, t_steps=t_steps, peak_tolerance=peak_tolerance
    )

    qc, metadata = build_quantum_walk_circuit(
        ctx["N"], ctx["M"], ctx["occupied_coords"], lam=lam, t_star=t_star,
        gamma=gamma, mode=mode
    )
    sv = Statevector.from_instruction(qc)
    probs = index_probs_from_statevector(sv, metadata["W"], metadata["IDX"])
    p_valid_circuit = float(np.sum(probs[valid_indices]))
    assert abs(p_valid_circuit - p_star) <= 1e-4, (
        f"Circuit/model mismatch: circuit={p_valid_circuit}, model={p_star}"
    )

    r_grover = np.pi / 4.0 * np.sqrt(W / max(1, len(valid_indices)))
    result = {
        "case": ctx["name"],
        "ctx": ctx,
        "qc": qc,
        "metadata": metadata,
        "A": A,
        "H_defect": H_defect,
        "lam": float(lam),
        "gamma": float(gamma),
        "mode": str(mode).lower(),
        "t_star": float(t_star),
        "P_valid": p_valid_circuit,
        "P_valid_model": float(p_star),
        "P_uniform": float(ctx["P_uniform"]),
        "probs": probs,
        "t_values": t_values,
        "p_valid_curve": p_valid_curve,
        "r_Grover": float(r_grover),
    }
    print(
        f"{ctx['name']} | {hamiltonian_mode_label(mode)} | W={W} | K={len(valid_indices)} | "
        f"lam*={float(lam):.4f} | t*={t_star:.4f} | "
        f"P_valid={p_valid_circuit:.4f} | vs Grover r={r_grover:.4f}"
    )
    return result


## Plotting Utilities for the Analysis Cells

These helper functions keep the analysis cells short while ensuring consistent colours, labels, figure sizes, and saved outputs.


In [9]:
def draw_time_evolution(ax, result, free_curve=None):
    """Draw P_valid(t) for the defect walk and optionally the free walk."""
    mode = result.get("mode", "adjacency")
    ax.plot(result["t_values"], result["p_valid_curve"], color=VALID_COLOR, linewidth=2.0, label=f"defect walk ({mode})")
    if free_curve is not None:
        t_free, p_free = free_curve
        ax.plot(t_free, p_free, color="black", linestyle="-.", linewidth=1.5, alpha=0.8, label=f"free walk ({mode})")
    ax.axvline(result["t_star"], color="black", linestyle=":", linewidth=1.5, label=f"t*={result['t_star']:.2f}")
    ax.axhline(result["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    ax.set_xlabel("time t")
    ax.set_ylabel("P_valid(t)")
    ax.legend(fontsize=10)


def free_hamiltonian_from_mode(A, gamma=1.0, mode="adjacency"):
    """Return the unmarked CTQW Hamiltonian for the selected convention."""
    # The free band must use the same convention as the defect Hamiltonian.
    mode = str(mode).lower()
    A = np.asarray(A, dtype=np.float64)
    if mode == "adjacency":
        return -float(gamma) * A.copy()
    if mode == "laplacian":
        L, _ = build_graph_laplacian(A)
        return float(gamma) * L.copy()
    raise ValueError(f"Unknown Hamiltonian mode: {mode!r}")


def spectrum_bound_state_data(result):
    """Compute defect eigenvalues and identify eigenvalues below the matching free-walk band."""
    # The reference band uses the same convention as the main Hamiltonian.
    H_defect = result["H_defect"]
    A = result["A"]
    gamma = result["gamma"]
    mode = result.get("mode", "adjacency")
    defect_eigs = np.linalg.eigvalsh(H_defect)
    free_eigs = np.linalg.eigvalsh(free_hamiltonian_from_mode(A, gamma=gamma, mode=mode))
    bulk_edge = float(np.min(free_eigs))
    bound_mask = defect_eigs < bulk_edge - 1e-8
    return defect_eigs, bulk_edge, bound_mask


def draw_spectrum(ax, result):
    """Draw the real-line eigenvalue spectrum of H_defect."""
    eigs, bulk_edge, bound_mask = spectrum_bound_state_data(result)
    y = np.zeros_like(eigs)
    ax.scatter(eigs[~bound_mask], y[~bound_mask], color=BASELINE_COLOR, s=38, label="bulk")
    if np.any(bound_mask):
        ax.scatter(eigs[bound_mask], y[bound_mask], color=VALID_COLOR, marker="*", s=160, label="bound state")
        nearest_bound = float(np.max(eigs[bound_mask]))
        gap = bulk_edge - nearest_bound
        ax.plot([nearest_bound, bulk_edge], [0.08, 0.08], color="black", linewidth=1.4)
        ax.text((nearest_bound + bulk_edge) / 2, 0.11, f"gap={gap:.3f}", ha="center", fontsize=10)
        ax.annotate("bound state", xy=(nearest_bound, 0.0), xytext=(nearest_bound, 0.25),
                    arrowprops={"arrowstyle": "->", "linewidth": 1.0}, fontsize=10)
    else:
        ax.text(0.02, 0.85, "no isolated bound state", transform=ax.transAxes, fontsize=10)
    ax.axvline(bulk_edge, color="black", linestyle="--", linewidth=1.0, label="free band edge")
    ax.set_yticks([])
    ax.set_xlabel("eigenvalue")
    ax.set_ylim(-0.2, 0.45)
    ax.legend(fontsize=9)


def draw_distribution(ax, result):
    """Draw the optimal index probability distribution with valid/invalid colours."""
    W = result["ctx"]["W"]
    valid = set(result["ctx"]["valid_indices"])
    colors = [VALID_COLOR if i in valid else INVALID_COLOR for i in range(W)]
    x = np.arange(W)
    ax.bar(x, result["probs"], color=colors, alpha=0.9)
    ax.axhline(1.0 / W, color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="1/W")
    ax.set_xlabel("window index i")
    ax.set_ylabel("probability")
    ax.legend(fontsize=10)


def best_lambda_from_scan(lambda_scan_rows, p_tolerance=V7_PEAK_TOLERANCE):
    """Choose the earliest near-optimal lambda-scan row.

    We first find the maximum scanned probability, then accept all lambdas
    within p_tolerance of it and choose the one with the smallest t_star.
    V7 uses this as a probability-time tradeoff, so we do not pay for late
    recurrence peaks that only improve P_valid slightly.
    """
    max_p = max(row[2] for row in lambda_scan_rows)
    candidates = [row for row in lambda_scan_rows if row[2] >= max_p - float(p_tolerance)]
    return min(candidates, key=lambda row: (row[1], -row[2], row[0]))


def v7_lambda_grid(W):
    """Return the V7 lambda scan grid used for the main defect runs."""
    # Mix geometric and linear points to capture resonances without making the cell slow.
    upper = 5.0 * np.sqrt(W)
    weak_to_mid = np.geomspace(0.05, upper, 32)
    linear = np.linspace(0.1, upper, 36)
    heuristic_band = np.sqrt(W) * np.linspace(0.25, 1.75, 20)
    return np.unique(np.r_[weak_to_mid, linear, heuristic_band])


def free_walk_curve(ctx, A, t_max=200, t_steps=4000, gamma=1.0, mode="adjacency"):
    """Return the lambda=0 time curve for the selected Hamiltonian convention."""
    H_free = build_defect_hamiltonian(
        A, ctx["valid_indices"], lam=0.0, gamma=gamma, mode=mode
    )
    t_star, p_star, t_values, p_curve = find_optimal_time(
        H_free, ctx["valid_indices"], ctx["W"], t_max=t_max, t_steps=t_steps, peak_tolerance=V7_PEAK_TOLERANCE
    )
    return {
        "t_star": float(t_star),
        "P_free": float(p_star),
        "t_values": t_values,
        "p_curve": p_curve,
    }


def grover_iteration_scale(W, K):
    """Return the usual Grover time/iteration scale (pi/4)*sqrt(W/K)."""
    # Use this scale as an optional bound when comparing time complexity.
    if K <= 0:
        raise ValueError("K must be positive for Grover scaling.")
    return float(np.pi / 4.0 * np.sqrt(float(W) / float(K)))


def grover_success_probability(W, K, r_value=None):
    """Return ideal continuous Grover success available within r_value.

    The textbook expression is sin^2((2r+1)theta), with
    sin(theta)=sqrt(K/W). For a theoretical face-off we allow the ideal
    continuous first peak if it lies within the Grover time scale; otherwise we
    report the endpoint probability. This avoids misleading overshoot artefacts
    for cases with a large valid fraction K/W.
    """
    # Compare against the ideal theoretical Grover limit within its time scale.
    if K <= 0:
        raise ValueError("K must be positive for Grover probability.")
    theta_g = np.arcsin(np.sqrt(float(K) / float(W)))
    if r_value is None:
        r_value = grover_iteration_scale(W, K)
    r_value = float(r_value)
    r_peak = max(0.0, np.pi / (4.0 * theta_g) - 0.5)
    r_eval = min(r_value, r_peak)
    return float(np.sin((2.0 * r_eval + 1.0) * theta_g) ** 2)


def case_grover_scale(ctx):
    """Return r_Grover for a case context."""
    return grover_iteration_scale(ctx["W"], ctx["K"])


def case_time_limit(ctx):
    """Return the active CTQW scan time limit for one case."""
    # If the bound is enabled, the walk cannot use more time than Grover.
    if V7_ENFORCE_GROVER_TIME_CAP:
        return case_grover_scale(ctx)
    return float(V7_DEFAULT_T_MAX)


def case_time_steps(ctx):
    """Return the number of sampled time points for one case."""
    # Keep high resolution even when t_max is small.
    return int(V7_T_STEPS)


def valid_geometry_summary(ctx, A):
    """Summarise how the valid set sits inside the window-start graph."""
    # Count internal valid edges and valid-invalid boundary edges.
    valid = set(ctx["valid_indices"])
    internal_valid_edges = 0
    boundary_edges = 0
    total_edges = 0
    degrees = A @ np.ones(A.shape[0])
    for i in range(ctx["W"]):
        for j in range(i + 1, ctx["W"]):
            if A[i, j] == 0:
                continue
            total_edges += 1
            if i in valid and j in valid:
                internal_valid_edges += 1
            elif (i in valid) != (j in valid):
                boundary_edges += 1
    valid_degrees = degrees[list(valid)] if valid else np.array([0.0])
    invalid = [i for i in range(ctx["W"]) if i not in valid]
    invalid_degrees = degrees[invalid] if invalid else np.array([0.0])
    return {
        "valid_fraction": ctx["K"] / ctx["W"],
        "internal_valid_edges": int(internal_valid_edges),
        "boundary_edges": int(boundary_edges),
        "total_edges": int(total_edges),
        "mean_valid_degree": float(np.mean(valid_degrees)),
        "mean_invalid_degree": float(np.mean(invalid_degrees)),
        "degree_bias_valid_minus_invalid": float(np.mean(valid_degrees) - np.mean(invalid_degrees)),
    }


## Lambda Scan for the Reference Case

The defect strength controls how strongly valid nodes are energetically separated from the rest of the graph. Here we scan $\lambda$ for case `02_1d_main_reference`, record the best success probability and best time for each value, and mark the optimal $\lambda^*$.


In [10]:
CASE02 = next(ctx for ctx in CASE_CONTEXTS if ctx["name"] == "02_1d_main_reference")
A_case02 = build_adjacency_matrix(CASE02["starts"], CASE02["N"])
lambda_values_case02 = v7_lambda_grid(CASE02["W"])
lambda_scan_case02 = scan_lambda(
    A_case02,
    CASE02["valid_indices"],
    CASE02["W"],
    lambda_values=lambda_values_case02,
    t_max=case_time_limit(CASE02),
    t_steps=case_time_steps(CASE02),
    mode="adjacency",
    peak_tolerance=V7_PEAK_TOLERANCE,
)
best_lam_case02, best_t_scan_case02, best_p_scan_case02 = best_lambda_from_scan(lambda_scan_case02)
r_grover_case02 = case_grover_scale(CASE02)

scan_lams = np.array([row[0] for row in lambda_scan_case02])
scan_tstars = np.array([row[1] for row in lambda_scan_case02])
scan_pstars = np.array([row[2] for row in lambda_scan_case02])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(scan_lams, scan_pstars, marker="o", color=VALID_COLOR, linewidth=2)
axes[0].axhline(CASE02["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
axes[0].axvline(best_lam_case02, color="black", linestyle=":", linewidth=1.5, label=f"lambda*={best_lam_case02:.3f}")
axes[0].set_xlabel("lambda")
axes[0].set_ylabel("P_valid(t*)")
axes[0].set_title("Adjacency convention: H=-gamma A-lambda P_V")
axes[0].legend(fontsize=10)

axes[1].plot(scan_lams, scan_tstars, marker="o", color=VALID_COLOR, linewidth=2)
axes[1].axhline(r_grover_case02, color=INVALID_COLOR, linestyle="--", linewidth=1.5, label="r_Grover")
axes[1].axvline(best_lam_case02, color="black", linestyle=":", linewidth=1.5, label=f"lambda*={best_lam_case02:.3f}")
axes[1].set_xlabel("lambda")
axes[1].set_ylabel("earliest near-optimal t_star")
axes[1].set_title(f"Time selected within {V7_PEAK_TOLERANCE:.2f} of peak")
axes[1].legend(fontsize=10)

fig.suptitle("Lambda scan — 02_1d_main_reference", fontsize=14)
fig.tight_layout()
save_qw_figure(fig, "v7_lambda_scan_case02")
print(f"Best lambda for case 02: lambda*={best_lam_case02:.6f}, t*={best_t_scan_case02:.6f}, P*={best_p_scan_case02:.6f}")


Best lambda for case 02: lambda*=1.975537, t*=1.128372, P*=0.675462


## Time Evolution Curve

For the optimal defect strength found above, we plot the full success-probability curve $P_{valid}(t)$. We also overlay the free walk with $\lambda=0$ to show why the defect is needed: without the local energy perturbation, the valid windows are not spectrally singled out.


In [11]:
rep_ctx = CASE02
rep_result = run_qw_case(rep_ctx, lam=best_lam_case02, gamma=1.0, mode="adjacency", t_max=case_time_limit(rep_ctx), t_steps=case_time_steps(rep_ctx), peak_tolerance=V7_PEAK_TOLERANCE)
free_case02 = free_walk_curve(rep_ctx, rep_result["A"], t_max=case_time_limit(rep_ctx), t_steps=case_time_steps(rep_ctx), mode="adjacency")
t_free, p_free = free_case02["t_values"], free_case02["p_curve"]

fig, ax = plt.subplots(figsize=(10, 5))
draw_time_evolution(ax, rep_result, free_curve=(t_free, p_free))
ax.set_title(f"Time evolution — {rep_ctx['name']}")
fig.tight_layout()
save_qw_figure(fig, f"v7_time_evolution_{qw_slug(rep_ctx['name'])}")


02_1d_main_reference | Adjacency convention: H = -gamma * A - lambda * P_V | W=7 | K=2 | lam*=1.9755 | t*=1.1284 | P_valid=0.6755 | vs Grover r=1.4693


## Eigenvalue Spectrum

The Hamiltonian is Hermitian, so its eigenvalues are real. A successful defect walk is associated with bound-state eigenvalues below the free-walk bulk band. These isolated eigenvalues explain why amplitude can localise around valid nodes.


In [12]:

fig, ax = plt.subplots(figsize=(10, 5))
draw_spectrum(ax, rep_result)
ax.set_title(f"Eigenvalue spectrum — {rep_ctx['name']}")
fig.tight_layout()
save_qw_figure(fig, f"v7_spectrum_{qw_slug(rep_ctx['name'])}")


## Optimal Probability Distribution

This bar chart shows the probability of measuring each candidate start index at the optimal time. Valid starts are green, invalid starts are red, and the dashed line is the uniform $1/W$ baseline.


In [13]:

fig, ax = plt.subplots(figsize=(10, 5))
draw_distribution(ax, rep_result)
ax.set_title(f"Optimal distribution — {rep_ctx['name']}, P_valid={rep_result['P_valid']:.4f}")
fig.tight_layout()
save_qw_figure(fig, f"v7_distribution_{qw_slug(rep_ctx['name'])}")


## 2x2 Summary Dashboard

The dashboard combines the key diagnostics for the representative case: time evolution, spectrum, optimal distribution, and lambda scan. This mirrors the compact comparison style used in V4.


In [14]:

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

draw_time_evolution(axes[0, 0], rep_result, free_curve=(t_free, p_free))
axes[0, 0].set_title("Time evolution")

draw_spectrum(axes[0, 1], rep_result)
axes[0, 1].set_title("Eigenvalue spectrum")

draw_distribution(axes[1, 0], rep_result)
axes[1, 0].set_title("Optimal distribution")

axes[1, 1].plot(scan_lams, scan_pstars, marker="o", color=VALID_COLOR, linewidth=2)
axes[1, 1].axhline(rep_ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
axes[1, 1].axvline(best_lam_case02, color="black", linestyle=":", linewidth=1.5, label=f"lambda*={best_lam_case02:.3f}")
axes[1, 1].set_xlabel("lambda")
axes[1, 1].set_ylabel("P_valid(t*)")
axes[1, 1].set_title("Lambda scan")
axes[1, 1].legend(fontsize=9)

fig.suptitle(f"QW defect summary — {rep_ctx['name']}", fontsize=14)
fig.tight_layout()
save_qw_figure(fig, f"v7_summary_{qw_slug(rep_ctx['name'])}")


## Cross-Case Comparison

We now apply the same lambda-scan procedure to all ten V4 cases. For each case, we choose the scanned $\lambda$ with the highest $P_{valid}(t^*)$, run the exact circuit, and compare the final success probability to the uniform baseline and Grover iteration scale.


In [15]:
V7_MAIN_MODE = "adjacency"
V7_COMPARISON_MODES = ("adjacency", "laplacian")

ALL_MODE_RESULTS = {mode: [] for mode in V7_COMPARISON_MODES}
ALL_MODE_SCANS = {mode: {} for mode in V7_COMPARISON_MODES}

for mode in V7_COMPARISON_MODES:
    print()
    print(f"Running V7 mode: {hamiltonian_mode_label(mode)}")
    for ctx in CASE_CONTEXTS:
        A_ctx = build_adjacency_matrix(ctx["starts"], ctx["N"])
        lambda_values = v7_lambda_grid(ctx["W"])
        scan_rows = scan_lambda(
            A_ctx,
            ctx["valid_indices"],
            ctx["W"],
            lambda_values=lambda_values,
            t_max=case_time_limit(ctx),
            t_steps=case_time_steps(ctx),
            mode=mode,
            peak_tolerance=V7_PEAK_TOLERANCE,
        )
        ALL_MODE_SCANS[mode][ctx["name"]] = scan_rows
        best_lam, _best_t, _best_p = best_lambda_from_scan(scan_rows, p_tolerance=V7_PEAK_TOLERANCE)
        result = run_qw_case(
            ctx, lam=best_lam, gamma=1.0, mode=mode, t_max=case_time_limit(ctx), t_steps=case_time_steps(ctx),
            peak_tolerance=V7_PEAK_TOLERANCE,
        )

        free = free_walk_curve(ctx, A_ctx, t_max=case_time_limit(ctx), t_steps=case_time_steps(ctx), mode=mode)
        geom = valid_geometry_summary(ctx, A_ctx)

        r_grover = case_grover_scale(ctx)
        p_grover = grover_success_probability(ctx["W"], ctx["K"], r_grover)

        result.update({
            "time_cap_enabled": bool(V7_ENFORCE_GROVER_TIME_CAP),
            "time_limit": case_time_limit(ctx),
            "P_Grover_theory": p_grover,
            "r_Grover": r_grover,
            "P_free": free["P_free"],
            "t_free": free["t_star"],
            "free_t_values": free["t_values"],
            "free_curve": free["p_curve"],
            "P_gain_over_uniform": result["P_valid"] - result["P_uniform"],
            "P_gain_over_free": result["P_valid"] - free["P_free"],
            "lambda_at_lower_bound": abs(result["lam"] - float(lambda_values[0])) < 1e-12,
            **geom,
        })
        ALL_MODE_RESULTS[mode].append(result)

ALL_QW_RESULTS = ALL_MODE_RESULTS[V7_MAIN_MODE]
ALL_LAMBDA_SCANS = ALL_MODE_SCANS[V7_MAIN_MODE]
ADJACENCY_RESULTS = ALL_MODE_RESULTS["adjacency"]
LAPLACIAN_RESULTS = ALL_MODE_RESULTS["laplacian"]

case_labels = [res["case"].split("_", 1)[0] for res in ALL_QW_RESULTS]
x = np.arange(len(ALL_QW_RESULTS))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(max(14, 0.75 * len(ALL_QW_RESULTS)), 5))

uniform_vals = np.array([res["P_uniform"] for res in ALL_QW_RESULTS])
qw_vals = np.array([res["P_valid"] for res in ALL_QW_RESULTS])
axes[0].bar(x - width/2, uniform_vals, width=width, color=BASELINE_COLOR, alpha=0.75, label="P_uniform")
axes[0].bar(x + width/2, qw_vals, width=width, color=VALID_COLOR, alpha=0.9, label="P_valid adjacency QW")
for xi, value in zip(x, qw_vals):
    axes[0].text(xi + width/2, value + 0.015, f"{value:.2f}", ha="center", va="bottom", fontsize=9, rotation=90)
axes[0].set_xticks(x)
axes[0].set_xticklabels(case_labels, rotation=45, ha="right")
axes[0].set_ylim(0.0, min(1.15, max(1.0, qw_vals.max() + 0.12)))
axes[0].set_xlabel("case")
axes[0].set_ylabel("success probability")
axes[0].set_title("Uniform baseline vs adjacency QW")
axes[0].legend(fontsize=10)

r_grover_vals = np.array([res["r_Grover"] for res in ALL_QW_RESULTS])
t_star_vals = np.array([res["t_star"] for res in ALL_QW_RESULTS])
axes[1].plot(x, r_grover_vals, marker="o", color=INVALID_COLOR, linewidth=2, label="r_Grover")
axes[1].plot(x, t_star_vals, marker="o", color=VALID_COLOR, linewidth=2, label="adjacency t_star")
axes[1].set_xticks(x)
axes[1].set_xticklabels(case_labels, rotation=45, ha="right")
axes[1].set_xlabel("case")
axes[1].set_ylabel("time / iteration scale")
axes[1].set_title("Grover scale vs adjacency QW time")
axes[1].legend(fontsize=10)

fig.tight_layout()
save_qw_figure(fig, "v7_cross_case_comparison")



Running V7 mode: Adjacency convention: H = -gamma * A - lambda * P_V
01_1d_tiny_single_gap | Adjacency convention: H = -gamma * A - lambda * P_V | W=5 | K=1 | lam*=4.2156 | t*=0.5107 | P_valid=0.4929 | vs Grover r=1.7562
02_1d_main_reference | Adjacency convention: H = -gamma * A - lambda * P_V | W=7 | K=2 | lam*=1.9755 | t*=1.1284 | P_valid=0.6755 | vs Grover r=1.4693


03_1d_two_free_regions | Adjacency convention: H = -gamma * A - lambda * P_V | W=8 | K=3 | lam*=2.5072 | t*=0.8993 | P_valid=0.6494 | vs Grover r=1.2825
04_1d_clustered_medium | Adjacency convention: H = -gamma * A - lambda * P_V | W=14 | K=4 | lam*=2.7670 | t*=0.8579 | P_valid=0.5892 | vs Grover r=1.4693


05_1d_long_clustered_blocks | Adjacency convention: H = -gamma * A - lambda * P_V | W=29 | K=11 | lam*=3.1658 | t*=0.6413 | P_valid=0.5392 | vs Grover r=1.2752
06_2d_tiny_corner_block | Adjacency convention: H = -gamma * A - lambda * P_V | W=4 | K=1 | lam*=3.2114 | t*=0.8539 | P_valid=0.7953 | vs Grover r=1.5708
07_2d_small_diagonal_block | Adjacency convention: H = -gamma * A - lambda * P_V | W=9 | K=1 | lam*=3.1184 | t*=1.7782 | P_valid=0.7033 | vs Grover r=2.3562


08_2d_medium_clustered_obstacles | Adjacency convention: H = -gamma * A - lambda * P_V | W=16 | K=9 | lam*=3.5263 | t*=0.4957 | P_valid=0.8405 | vs Grover r=1.0472


09_2d_rectangular_window | Adjacency convention: H = -gamma * A - lambda * P_V | W=20 | K=8 | lam*=3.9161 | t*=0.6940 | P_valid=0.8120 | vs Grover r=1.2418


...

16_2d_narrow_passages | Laplacian convention: H = gamma * L - lambda * P_V | W=42 | K=15 | lam*=2.8689 | t*=0.8896 | P_valid=0.7664 | vs Grover r=1.3142


17_2d_large_clustered_office | Laplacian convention: H = gamma * L - lambda * P_V | W=49 | K=13 | lam*=3.0914 | t*=1.0604 | P_valid=0.6960 | vs Grover r=1.5248


18_2d_rectangular_window_clusters | Laplacian convention: H = gamma * L - lambda * P_V | W=45 | K=17 | lam*=3.3311 | t*=0.7292 | P_valid=0.8101 | vs Grover r=1.2778


19_3d_dual_room_clusters | Laplacian convention: H = gamma * L - lambda * P_V | W=48 | K=26 | lam*=4.4669 | t*=0.4566 | P_valid=0.8414 | vs Grover r=1.0671


20_3d_layered_obstacle_bands | Laplacian convention: H = gamma * L - lambda * P_V | W=60 | K=32 | lam*=4.9941 | t*=0.3972 | P_valid=0.8369 | vs Grover r=1.0755


## Per-Case Dashboards

The previous cells focused on the representative 1D case. This cell creates the same summary diagnostics for every case: lambda scan, time evolution with free-walk overlay, spectrum, and final probability distribution. These figures make it easier to see whether each instance is improved by the defect or mostly by free graph evolution.


In [16]:
for result in ALL_QW_RESULTS:
    ctx = result["ctx"]
    slug = qw_slug(ctx["name"])
    scan_rows = ALL_LAMBDA_SCANS[ctx["name"]]
    lams = np.array([row[0] for row in scan_rows])
    pstars = np.array([row[2] for row in scan_rows])
    tstars = np.array([row[1] for row in scan_rows])

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    axes[0, 0].plot(lams, pstars, marker="o", color=VALID_COLOR, linewidth=2)
    axes[0, 0].axhline(ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    axes[0, 0].axhline(result["P_free"], color="black", linestyle="-.", linewidth=1.3, label="free max")
    axes[0, 0].axvline(result["lam"], color="black", linestyle=":", linewidth=1.5, label=f"lambda*={result['lam']:.3f}")
    axes[0, 0].set_xlabel("lambda")
    axes[0, 0].set_ylabel("P_valid(t*)")
    axes[0, 0].set_title("Adjacency convention scan")
    axes[0, 0].legend(fontsize=8)

    draw_time_evolution(axes[0, 1], result, free_curve=(result["free_t_values"], result["free_curve"]))
    axes[0, 1].set_title("Time evolution")

    draw_distribution(axes[1, 0], result)
    axes[1, 0].set_title(f"Distribution, P={result['P_valid']:.4f}")

    draw_spectrum(axes[1, 1], result)
    axes[1, 1].set_title("Spectrum")

    fig.suptitle(f"Adjacency QW defect per-case summary — {ctx['name']}", fontsize=14)
    fig.tight_layout()
    save_qw_figure(fig, f"v7_case_summary_{slug}")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(lams, pstars, marker="o", color=VALID_COLOR, linewidth=2, label="adjacency convention")
    ax.axhline(ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    ax.axhline(result["P_free"], color="black", linestyle="-.", linewidth=1.4, label="free walk max")
    ax.axvline(result["lam"], color="black", linestyle=":", linewidth=1.5, label=f"lambda*={result['lam']:.3f}")
    ax.set_xlabel("lambda")
    ax.set_ylabel("P_valid(t*)")
    ax.set_title(f"Lambda scan — {ctx['name']}")
    ax.legend(fontsize=10)
    fig.tight_layout()
    save_qw_figure(fig, f"v7_lambda_scan_{slug}")

    fig, ax = plt.subplots(figsize=(10, 5))
    draw_time_evolution(ax, result, free_curve=(result["free_t_values"], result["free_curve"]))
    ax.set_title(f"Time evolution — {ctx['name']}")
    fig.tight_layout()
    save_qw_figure(fig, f"v7_time_evolution_{slug}")

    fig, ax = plt.subplots(figsize=(10, 5))
    draw_distribution(ax, result)
    ax.set_title(f"Optimal distribution — {ctx['name']}, P_valid={result['P_valid']:.4f}")
    fig.tight_layout()
    save_qw_figure(fig, f"v7_distribution_{slug}")

    fig, ax = plt.subplots(figsize=(10, 5))
    draw_spectrum(ax, result)
    ax.set_title(f"Eigenvalue spectrum — {ctx['name']}")
    fig.tight_layout()
    save_qw_figure(fig, f"v7_spectrum_{slug}")

print(f"Saved adjacency per-case V7 analysis figures for {len(ALL_QW_RESULTS)} cases.")


Saved adjacency per-case V7 analysis figures for 20 cases.


## All-Case Diagnostic Figures

These all-case plots compare the uniform baseline, the free walk, the corrected `-A - lambda P` defect walk, and the previous `+A - lambda P` convention. The goal is to verify that the fix improves the probabilities and reduces the chosen times by avoiding late recurrences.


In [17]:
case_labels = [res["case"].split("_", 1)[0] for res in ALL_QW_RESULTS]
x = np.arange(len(ALL_QW_RESULTS))

uniform_vals = np.array([res["P_uniform"] for res in ADJACENCY_RESULTS])
adj_free_vals = np.array([res["P_free"] for res in ADJACENCY_RESULTS])
adj_vals = np.array([res["P_valid"] for res in ADJACENCY_RESULTS])
lap_vals = np.array([res["P_valid"] for res in LAPLACIAN_RESULTS])
adj_t = np.array([res["t_star"] for res in ADJACENCY_RESULTS])
lap_t = np.array([res["t_star"] for res in LAPLACIAN_RESULTS])

fig, axes = plt.subplots(1, 2, figsize=(max(14, 0.75 * len(ALL_QW_RESULTS)), 5))
width = 0.22
axes[0].bar(x - 1.5*width, uniform_vals, width=width, color=BASELINE_COLOR, alpha=0.75, label="uniform")
axes[0].bar(x - 0.5*width, adj_free_vals, width=width, color="black", alpha=0.65, label="free adjacency")
axes[0].bar(x + 0.5*width, adj_vals, width=width, color=VALID_COLOR, alpha=0.9, label="Adjacency convention")
axes[0].bar(x + 1.5*width, lap_vals, width=width, color="#3498db", alpha=0.85, label="Laplacian convention")
axes[0].set_xticks(x)
axes[0].set_xticklabels(case_labels, rotation=45, ha="right")
axes[0].set_ylabel("best P_valid")
axes[0].set_xlabel("case")
axes[0].set_title("Probability: adjacency vs Laplacian")
axes[0].legend(fontsize=9)

axes[1].bar(x - width/2, adj_t, width=width, color=VALID_COLOR, alpha=0.9, label="Adjacency t*")
axes[1].bar(x + width/2, lap_t, width=width, color="#3498db", alpha=0.85, label="Laplacian t*")
axes[1].plot(x, [res["r_Grover"] for res in ADJACENCY_RESULTS], color=INVALID_COLOR, marker="o", linewidth=1.8, label="r_Grover")
axes[1].set_xticks(x)
axes[1].set_xticklabels(case_labels, rotation=45, ha="right")
axes[1].set_ylabel("time / iteration scale")
axes[1].set_xlabel("case")
axes[1].set_title("Time: adjacency vs Laplacian")
axes[1].legend(fontsize=9)
fig.suptitle("Hamiltonian convention comparison", fontsize=14)
fig.tight_layout()
save_qw_figure(fig, "v7_hamiltonian_convention_comparison")

n_scan_cases = len(ALL_QW_RESULTS)
n_scan_cols = min(5, n_scan_cases)
n_scan_rows = int(np.ceil(n_scan_cases / n_scan_cols))
fig, axes = plt.subplots(
    n_scan_rows,
    n_scan_cols,
    figsize=(max(14, 2.8 * n_scan_cols), max(7, 2.6 * n_scan_rows)),
    sharey=True,
)
axes_grid = np.asarray(axes).reshape(n_scan_rows, n_scan_cols)
axes_flat = axes_grid.ravel()
for ax, result in zip(axes_flat, ALL_QW_RESULTS):
    scan_rows = ALL_LAMBDA_SCANS[result["case"]]
    lams = np.array([row[0] for row in scan_rows])
    pstars = np.array([row[2] for row in scan_rows])
    ax.plot(lams, pstars, color=VALID_COLOR, linewidth=1.8, label="adjacency")
    lap_scan = ALL_MODE_SCANS["laplacian"][result["case"]]
    lap_lams = np.array([row[0] for row in lap_scan])
    lap_pstars = np.array([row[2] for row in lap_scan])
    ax.plot(lap_lams, lap_pstars, color="#3498db", linewidth=1.4, linestyle="--", label="laplacian")
    ax.axhline(result["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.0)
    ax.axvline(result["lam"], color="black", linestyle=":", linewidth=1.0)
    ax.set_title(result["case"].split("_", 1)[0], fontsize=11)
    ax.set_xlabel("lambda")
for ax in axes_flat[n_scan_cases:]:
    ax.axis("off")
for row in range(n_scan_rows):
    axes_grid[row, 0].set_ylabel("P_valid(t*)")
axes_grid[0, 0].legend(fontsize=8)
fig.suptitle("All lambda scans — adjacency and Laplacian conventions", fontsize=14)
fig.tight_layout()
save_qw_figure(fig, "v7_all_lambda_scans")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
valid_fraction = np.array([res["valid_fraction"] for res in ADJACENCY_RESULTS])
degree_bias = np.array([res["degree_bias_valid_minus_invalid"] for res in ADJACENCY_RESULTS])
gain_vs_uniform = adj_vals - uniform_vals
gain_vs_free = adj_vals - adj_free_vals

axes[0].scatter(valid_fraction, gain_vs_uniform, s=80, color=VALID_COLOR, label="adjacency")
axes[0].scatter(valid_fraction, lap_vals - uniform_vals, s=80, color="#3498db", marker="s", label="laplacian")
for label, xf, yf in zip(case_labels, valid_fraction, gain_vs_uniform):
    axes[0].annotate(label, (xf, yf), textcoords="offset points", xytext=(4, 4), fontsize=9)
axes[0].axhline(0.0, color="black", linewidth=1.0)
axes[0].set_xlabel("valid fraction K/W")
axes[0].set_ylabel("P_QW - P_uniform")
axes[0].set_title("Gain vs valid-set size")
axes[0].legend(fontsize=9)

axes[1].scatter(degree_bias, adj_vals - lap_vals, s=80, color="#8e44ad")
for label, xf, yf in zip(case_labels, degree_bias, adj_vals - lap_vals):
    axes[1].annotate(label, (xf, yf), textcoords="offset points", xytext=(4, 4), fontsize=9)
axes[1].axhline(0.0, color="black", linewidth=1.0)
axes[1].set_xlabel("mean degree(valid) - mean degree(invalid)")
axes[1].set_ylabel("P_adjacency - P_laplacian")
axes[1].set_title("Why the conventions differ on finite graphs")
fig.tight_layout()
save_qw_figure(fig, "v7_gain_diagnostics")

fig, axes = plt.subplots(1, 2, figsize=(max(14, 0.75 * len(ALL_QW_RESULTS)), 5))
axes[0].bar(x - width/2, lap_t, width=width, color="#3498db", alpha=0.85, label="Laplacian")
axes[0].bar(x + width/2, adj_t, width=width, color=VALID_COLOR, alpha=0.9, label="Adjacency")
axes[0].set_xticks(x)
axes[0].set_xticklabels(case_labels, rotation=45, ha="right")
axes[0].set_ylabel("selected t_star")
axes[0].set_xlabel("case")
axes[0].set_title("Selected walk time after earliest-peak rule")
axes[0].legend(fontsize=9)

axes[1].scatter(adj_t, adj_vals, s=80, color=VALID_COLOR, label="adjacency")
axes[1].scatter(lap_t, lap_vals, s=80, color="#3498db", marker="s", label="laplacian")
for label, xf, yf in zip(case_labels, adj_t, adj_vals):
    axes[1].annotate(label, (xf, yf), textcoords="offset points", xytext=(4, 4), fontsize=9)
axes[1].set_xlabel("t_star")
axes[1].set_ylabel("P_valid")
axes[1].set_title("Probability-time tradeoff")
axes[1].legend(fontsize=9)
fig.tight_layout()
save_qw_figure(fig, "v7_time_reduction_diagnostics")

fig, axes = plt.subplots(1, 2, figsize=(max(14, 0.75 * len(ALL_QW_RESULTS)), 5))
grover_p = np.array([res["P_Grover_theory"] for res in ADJACENCY_RESULTS])
grover_t = np.array([res["r_Grover"] for res in ADJACENCY_RESULTS])

axes[0].bar(x - width/2, grover_p, width=width, color=INVALID_COLOR, alpha=0.75, label="Grover theoretical")
axes[0].bar(x + width/2, adj_vals, width=width, color=VALID_COLOR, alpha=0.9, label="V7 adjacency QW")
axes[0].set_xticks(x)
axes[0].set_xticklabels(case_labels, rotation=45, ha="right")
axes[0].set_ylim(0.0, 1.08)
axes[0].set_xlabel("case")
axes[0].set_ylabel("success probability")
axes[0].set_title("Probability face-off")
axes[0].legend(fontsize=9)

axes[1].bar(x - width/2, grover_t, width=width, color=INVALID_COLOR, alpha=0.75, label="Grover r_Grover")
axes[1].bar(x + width/2, adj_t, width=width, color=VALID_COLOR, alpha=0.9, label="V7 adjacency QW t*")
axes[1].set_xticks(x)
axes[1].set_xticklabels(case_labels, rotation=45, ha="right")
axes[1].set_xlabel("case")
axes[1].set_ylabel("time / iteration scale")
axes[1].set_title(
    "Time face-off" + (" (Grover cap ON)" if V7_ENFORCE_GROVER_TIME_CAP else " (Grover cap OFF)")
)
axes[1].legend(fontsize=9)
fig.suptitle("V7 quantum walk vs Grover theoretical", fontsize=14)
fig.tight_layout()
save_qw_figure(fig, "v7_qw_vs_grover_faceoff")

print("Convention comparison:")
print("  1. Adjacency convention: H=-gamma A-lambda P_V.")
print("  2. Laplacian convention: H=gamma L-lambda P_V.")
print("  3. On non-regular finite window-start graphs, L=D-A adds degree-dependent diagonal phases, so the two modes can differ.")


Convention comparison:
  1. Adjacency convention: H=-gamma A-lambda P_V.
  2. Laplacian convention: H=gamma L-lambda P_V.
  3. On non-regular finite window-start graphs, L=D-A adds degree-dependent diagonal phases, so the two modes can differ.


## Comparison Table

The final table reports the main numerical quantities for every case: search-space size, number of valid windows, uniform baseline, optimal defect strength, optimal time, quantum-walk success probability, and Grover iteration scale. The same data are saved as CSV.


In [18]:
headers = [
    "case", "mode", "W", "K", "P_uniform", "lambda*", "t*", "P_valid(QW)",
    "time_cap_enabled", "time_limit", "P_Grover_theory", "P_free", "gain_vs_uniform", "gain_vs_free", "lambda_at_lower_bound",
    "r_Grover", "valid_fraction", "degree_bias_valid_minus_invalid",
]
rows = []
for mode in V7_COMPARISON_MODES:
    for res in ALL_MODE_RESULTS[mode]:
        ctx = res["ctx"]
        rows.append({
            "case": res["case"],
            "mode": res["mode"],
            "W": ctx["W"],
            "K": ctx["K"],
            "P_uniform": res["P_uniform"],
            "lambda*": res["lam"],
            "t*": res["t_star"],
            "P_valid(QW)": res["P_valid"],
            "time_cap_enabled": res["time_cap_enabled"],
            "time_limit": res["time_limit"],
            "P_Grover_theory": res["P_Grover_theory"],
            "P_free": res["P_free"],
            "gain_vs_uniform": res["P_gain_over_uniform"],
            "gain_vs_free": res["P_gain_over_free"],
            "lambda_at_lower_bound": res["lambda_at_lower_bound"],
            "r_Grover": res["r_Grover"],
            "valid_fraction": res["valid_fraction"],
            "degree_bias_valid_minus_invalid": res["degree_bias_valid_minus_invalid"],
        })

print(" | ".join(f"{h:>18s}" for h in headers))
print("-" * (21 * len(headers)))
for row in rows:
    print(
        f"{row['case']:>18s} | {row['mode']:>18s} | {row['W']:18d} | {row['K']:18d} | "
        f"{row['P_uniform']:18.6f} | {row['lambda*']:18.6f} | "
        f"{row['t*']:18.6f} | {row['P_valid(QW)']:18.6f} | "
        f"{str(row['time_cap_enabled']):>18s} | {row['time_limit']:18.6f} | "
        f"{row['P_Grover_theory']:18.6f} | {row['P_free']:18.6f} | {row['gain_vs_uniform']:18.6f} | "
        f"{row['gain_vs_free']:18.6f} | {str(row['lambda_at_lower_bound']):>18s} | "
        f"{row['r_Grover']:18.6f} | {row['valid_fraction']:18.6f} | "
        f"{row['degree_bias_valid_minus_invalid']:18.6f}"
    )

csv_path = V7_OUTPUT_DIR / "v7_results.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(rows)
print(f"Saved {csv_path}")


              case |               mode |                  W |                  K |          P_uniform |            lambda* |                 t* |        P_valid(QW) |   time_cap_enabled |         time_limit |    P_Grover_theory |             P_free |    gain_vs_uniform |       gain_vs_free | lambda_at_lower_bound |           r_Grover |     valid_fraction | degree_bias_valid_minus_invalid
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
01_1d_tiny_single_gap |          adjacency |                  5 |                  1 |           0.200000 |           4.215555 |           0.510744 |           0.492936 |               True |           1.756204 |           1.0000